# Phase 5 — Adaptive Agentic Response Planning

This engineering notebook adds bounded, local response planning and validation over the stable Phase 4 evidence engine.

## 1. Objective
Improve grounded answer structure, validation, and decision usefulness without replacing retrieval.

## 2. Phase 4 recap
Phase 4 owns recursive loading, hybrid retrieval, BM25, dense search, RRF, reranking, evidence selection, unsupported-query protection, and exports.

## 3. Why agentic response planning
Specialized review stages make planning and validation independently inspectable and testable.

## 4. Limitations of prompt-only generation
A single prompt does not provide explicit critic, compliance, risk, verification, or consensus contracts.

## 5. Agent architecture
```mermaid
flowchart LR
P4[Phase 4 evidence] --> QA[Query Analyzer] --> RP[Response Planner] --> PC[Prompt Composer] --> DG[Draft]
DG --> C[Critic]
DG --> CO[Compliance]
DG --> R[Risk]
DG --> EV[Evidence Verifier]
C --> CE[Consensus]
CO --> CE
R --> CE
EV --> CE
CE -->|at most once| DG
```

## 6. Multi-model local agent design
Capability tags (`text`, `vision`, `embedding`, `reranking`, `structured_json`) prevent incompatible routing. All clients are local and injected.

In [ ]:
phase5_config = {
    "phase5": {
        "enabled": True,
        "max_revision_loops": 1,
        "model_profiles": {
            "fast_text": {"provider": "ollama", "model": "qwen2.5:7b-instruct", "capabilities": ["text", "structured_json"]},
            "reasoning_text": {"provider": "ollama", "model": "phi4:14b", "capabilities": ["text", "structured_json"]},
            "vision_local": {"provider": "ollama", "model": "llava:13b", "capabilities": ["vision", "text"]},
        },
        "agents": {
            "query_analyzer": {"model_profile": "fast_text"},
            "response_planner": {"model_profile": "fast_text"},
            "draft_generator": {"model_profile": "reasoning_text"},
            "critic_agent": {"model_profile": "reasoning_text"},
            "compliance_agent": {"model_profile": "fast_text"},
            "risk_agent": {"model_profile": "reasoning_text"},
            "evidence_verifier": {"model_profile": "fast_text", "vision_model_profile": "vision_local"},
        },
    }
}

## 7. Query analysis
Classifies intent and determines whether risk, compliance, current data, or additional answer depth is required.

In [ ]:
# Optional real local-model demonstration; safely skips when Ollama or the profile is unavailable.
try:
    import ollama
    from cial_knowledge_os.agents import AgentState, ModelRouter, QueryAnalyzer
    installed = {item.model for item in ollama.list().models}
    if "qwen2.5:7b-instruct" in installed:
        real_analysis = QueryAnalyzer(ModelRouter(phase5_config)).run(
            AgentState("What cybersecurity controls should airport operations prioritize?")
        )
        display(real_analysis.output)
    else:
        print("Configured local model is not installed; real-model demo skipped.")
except Exception as exc:
    print(f"Ollama unavailable; real-model demo skipped: {exc}")

## 8. Response planning
Selects an enterprise answer format, sections, citation strategy, inclusions, and exclusions based on intent and selected evidence.

## 9. Prompt composition
Deterministically serializes only Phase 4 selected evidence, including optional modality fields. It never retrieves.

## 10. Draft generation
Generates exactly one grounded draft and preserves unsupported, insufficient-evidence, and generation-failed statuses.

## 11. Critic agent
Critiques completeness, structure, caveats, repetition, reasoning, prioritization, and unanswered parts without rewriting.

## 12. Compliance agent
Checks unsupported claims, citation discipline, grounding, weak-evidence disclosure, and enterprise caution.

## 13. Risk agent
Assesses operational, cybersecurity, aviation-safety, governance, and compliance risk with likelihood and mitigation status.

## 14. Evidence verifier
Citation checks are deterministic-first. Vision review is optional and reports `not_applicable` when no visual evidence exists.

## 15. Consensus engine
The deterministic decision is `accept`, `revise_once`, or `reject`; the configured revision count is hard-capped at one.

## 16. Single-query walkthrough
Construct the Phase 4 pipeline as in notebook 04, inject it into `Phase5Pipeline`, and call `answer(question)`. No Phase 4 code is duplicated.

In [ ]:
from cial_knowledge_os.agents import AgentState, Evidence
mock_state = AgentState(
    question="Which controls should be prioritized?",
    phase4_answer_status="answered",
    selected_evidence=[
        Evidence(evidence_id="text-1", source="manual.pdf", page=12, content="Prioritize tested controls.", score=0.82),
        Evidence(evidence_id="fig-1", source="architecture.pdf", page=4, modality="figure", image_path="figures/architecture.png", caption="Control architecture"),
    ],
)
mock_state.to_dict()

## 17. Multi-agent trace
`Phase5Trace` records model/profile, fallback, latency, success, warnings, errors, token estimate, and revision number for every stage.

In [ ]:
# Optional end-to-end real-agent answer over a deterministic Phase 4 fixture.
# It exercises local models without requiring the corpus to contain images.
try:
    import copy, ollama
    from types import SimpleNamespace
    from cial_knowledge_os import ModelRouter, Phase5Pipeline
    class NotebookPhase4Fixture:
        metrics = {}
        config = SimpleNamespace()
        def answer(self, question):
            return {"answer": "Prioritize tested controls [1].", "raw_answer": "Prioritize tested controls [1].", "answer_status": "answered", "selected_evidence": [{"text": "Prioritize controls that have been tested against operational requirements.", "source": "manual.pdf", "page_number": 12, "chunk_id": "c1", "reranker_score": 0.82}], "citations": [], "evidence_quality": {}}
    installed = {item.model for item in ollama.list().models}
    if "qwen2.5:7b-instruct" in installed and "phi4:14b" in installed:
        demo_config = copy.deepcopy(phase5_config)
        demo_config["phase5"]["max_revision_loops"] = 0
        real_phase5_answer = Phase5Pipeline(phase4_pipeline=NotebookPhase4Fixture(), config=demo_config, model_router=ModelRouter(demo_config)).answer(mock_state.question)
        display({key: real_phase5_answer[key] for key in ("answer", "final_status", "consensus_decision", "agent_latency_total_ms")})
    else:
        print("Configured local models are unavailable; end-to-end demo skipped.")
except Exception as exc:
    print(f"End-to-end local-model demo skipped: {exc}")

## 18. Manual QA
Review final claims against cited passages, verify visual paths locally, inspect high risks and uncited major claims, and record reviewer disposition.

## 19. Smoke comparison with Phase 4
Compare the same question's status, citations, selected evidence, answer structure, latency, and unsupported claims. Retrieval results must remain unchanged.

## 20. Benchmark readiness
Calibrate readiness and verification thresholds on versioned CIAL questions before making qualification claims.

## 21. Diagnostics
The Decision Intelligence Dashboard answers trust, evidence strength, unsupported claim, risk, consensus, revision, source, and enterprise-readiness questions.

In [ ]:
from cial_knowledge_os.reporting.phase5_html import render_phase5_html
# Mock dashboard; replace with one real Phase 5 result when local models are available.
mock_answer = {
    "question": mock_state.question, "answer": "Prioritize tested controls [1].",
    "selected_evidence": [item.to_dict() for item in mock_state.selected_evidence],
    "critic_review": {"passed": True, "severity": "low", "issues": []},
    "compliance_review": {"passed": True},
    "risk_review": {"passed": True, "risk_level": "low", "risks": []},
    "evidence_verification": {"passed": True, "verification_rate": 1.0, "verified_claims": ["Prioritize tested controls"], "unsupported_claims": [], "citation_mismatches": []},
    "consensus_decision": {"decision": "accept", "final_status": "answered"},
}
dashboard_html = render_phase5_html([mock_answer])
len(dashboard_html)

## 22. Advantages
Explicit contracts, independent tests, local model routing, deterministic consensus, multimodal preservation, bounded revision, and decision-focused reporting.

## 23. Limitations
Citation matching is not semantic entailment; local model quality varies; vision review depends on a configured local model; readiness scores require calibration.

## 24. Enterprise considerations
Keep inference offline, protect source paths, retain trace artifacts, calibrate thresholds, and require human authorization for safety-, security-, and compliance-sensitive actions.

## 25. Conclusion
Phase 5 improves how stable Phase 4 evidence becomes a validated decision-support answer. Set `phase5.enabled` to false or omit it to preserve Phase 4 behavior exactly.